In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import joblib
import warnings

warnings.filterwarnings('ignore')

# 注意：保持原始数据副本用于最后提交
train_raw = pd.read_csv('./train.csv')
test_raw = pd.read_csv('./test.csv')

train = train_raw.copy()
test = test_raw.copy()

print("训练集形状:", train.shape)
print("测试集形状:", test.shape)
print("\n训练集前5行:")
print(train.head())
print("\n缺失值统计:")
print(train.isnull().sum().sort_values(ascending=False).head(10))


def handle_missing(df, train_df=None):
    """处理缺失值逻辑"""
    # 第一阶段：删除高缺失率特征
    drop_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu']
    df = df.drop(drop_cols, axis=1)

    # 第二阶段：分组填充（只在训练模式时计算分组中位数）
    if train_df is not None:  # 测试集模式
        # 使用训练集计算的分组中位数
        grouped = train_df.groupby('MSSubClass')['LotFrontage'].median()
        df['LotFrontage'] = df['MSSubClass'].map(grouped).fillna(train_df['LotFrontage'].median())
    else:  # 训练集模式
        df['LotFrontage'] = df.groupby('MSSubClass')['LotFrontage'].transform(
            lambda x: x.fillna(x.median()))

    # 第三阶段：统一填充规则
    garage_cols = ['GarageType', 'GarageQual', 'GarageCond', 'GarageFinish']
    bsmt_cat_cols = ['BsmtFinType2', 'BsmtExposure', 'BsmtQual', 'BsmtFinType1', 'BsmtCond']

    # 分类特征填充'None'
    df[garage_cols] = df[garage_cols].fillna('None')
    df[bsmt_cat_cols] = df[bsmt_cat_cols].fillna('None')

    # 数值特征填充0
    df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)
    df['MasVnrArea'] = df['MasVnrArea'].fillna(0)

    # 删除剩余少量缺失行（仅训练集）
    if train_df is None:
        df = df.dropna(subset=['Electrical'])

    return df


# 处理训练集
train = handle_missing(train)
# 处理测试集（使用训练集统计量）
test = handle_missing(test, train)


def add_features(df):
    """添加组合特征"""
    # 时间相关特征
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['RemodelAge'] = df['YrSold'] - df['YearRemodAdd']

    # 组合面积特征
    df['TotalBsmtSF'] = df['BsmtFinSF1'] + df['BsmtFinSF2'] + df['BsmtUnfSF']
    df['TotalRooms'] = df['TotRmsAbvGrd'] + df['BedroomAbvGr'] + df['KitchenAbvGr']

    # 删除原始列
    drop_cols = ['YearBuilt', 'YearRemodAdd', 'YrSold', 'BsmtFinSF1', 'BsmtFinSF2',
                 'TotRmsAbvGrd', 'BedroomAbvGr', 'KitchenAbvGr']
    return df.drop(drop_cols, axis=1)


train = add_features(train)
test = add_features(test)

# 有序分类映射字典（新增MSZoning的映射）
ordinal_mappings = {
    'ExterQual': {'Ex':4, 'Gd':3, 'TA':2, 'Fa':1},
    'BsmtQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'None':1},
    'KitchenQual': {'Ex':4, 'Gd':3, 'TA':2, 'Fa':1},
    'MSZoning': {'RL':1, 'RM':2, 'C (all)':3, 'FV':4, 'RH':5}  # 新增
}

# 需要标签编码的特征（新增Utilities）
label_encode_cols = ['Street','LotShape','LandContour','Utilities','LandSlope']

def encode_features(df, train_df=None):
    """统一编码处理"""
    # 有序分类变量映射（新增MSZoning处理）
    for col in ['ExterQual', 'BsmtQual', 'KitchenQual', 'MSZoning']:  # 修改
        df[col] = df[col].map(ordinal_mappings[col])

    # 标签编码（添加异常处理）
    le = LabelEncoder()
    for col in label_encode_cols:
        if train_df is not None:  # 测试集模式
            # 填充训练集中未出现的类别为'Unknown'
            mask = ~df[col].isin(train_df[col].unique())
            df.loc[mask, col] = 'Unknown'
            le.fit(train_df[col].astype(str).tolist() + ['Unknown'])
        else:  # 训练集模式
            le.fit(df[col].astype(str))
        df[col] = le.transform(df[col].astype(str))

    # 独热编码（添加列对齐）
    ohe_cols = ['Neighborhood','HouseStyle']
    df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

    # 确保训练集和测试集列对齐（新增）
    if train_df is not None:
        train_columns = train_df.columns
        df = df.reindex(columns=train_columns, fill_value=0)

    return df

# 处理训练集时需要先拟合编码器
train = encode_features(train)
# 处理测试集时传入训练集参数
test = encode_features(test, train)  # 确保使用相同的编码规则

# 划分训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 使用随机森林进行特征重要性排序
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 获取特征重要性
feature_importances = pd.Series(rf.feature_importances_, index=X.columns)
top_features = feature_importances.nlargest(30).index.tolist()

# 筛选重要特征
X_train_sel = X_train[top_features]
X_val_sel = X_val[top_features]
test_sel = test[top_features]

# 特征标准化（基于训练集统计量）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_val_scaled = scaler.transform(X_val_sel)
test_scaled = scaler.transform(test_sel)


# 定义评估函数（在指数空间计算RMSE）
def exp_rmse(y_true, y_pred_log):
    y_true_exp = np.expm1(y_true)
    y_pred_exp = np.expm1(y_pred_log)
    return np.sqrt(mean_squared_error(y_true_exp, y_pred_exp))


# 随机森林模型
rf_model = RandomForestRegressor(random_state=42)

# 超参数搜索空间
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 15, 25],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# 随机搜索
search = RandomizedSearchCV(
    rf_model, param_dist, n_iter=15, cv=5,
    scoring='neg_mean_squared_error', n_jobs=-1
)
search.fit(X_train_scaled, y_train)

# 最佳模型评估
best_model = search.best_estimator_
val_pred = best_model.predict(X_val_scaled)
print(f"验证集RMSE: {exp_rmse(y_val, val_pred):.4f}")
print("最佳参数:", search.best_params_)

# 训练最终模型（使用全量数据）
X_full_scaled = scaler.transform(X[top_features])
best_model.fit(X_full_scaled, y)

# 预测测试集
test_pred_log = best_model.predict(test_scaled)
test_pred = np.expm1(test_pred_log)  # 指数变换

# 生成提交文件
submission = pd.DataFrame({
    'Id': test_raw['Id'],
    'SalePrice': test_pred
})
submission.to_csv('submission.csv', index=False)
print("提交文件已生成！")

# 保存预处理管道和模型
joblib.dump({
    'scaler': scaler,
    'top_features': top_features,
    'model': best_model
}, 'house_price_pipeline.pkl')

训练集形状: (1460, 81)
测试集形状: (1459, 80)

训练集前5行:
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  Sal

ValueError: could not convert string to float: 'RL'